# Huấn luyện TF-IDF và thử từ khóa

Notebook này tự chứa code xử lý. Chỉ cần trỏ thư mục dữ liệu, rồi chạy lần lượt các ô.

1. Ô cấu hình: sửa `DATA_DIR` nếu file `.txt` không nằm ở `data/corpus`. Trên Colab, Drive được gắn sẵn và `DATA_DIR` trỏ tới `/content/drive/MyDrive/CTH625_Nhom3/data/corpus`.
2. Các ô tiếp theo định nghĩa cách đọc file, tiền xử lý, TF-IDF, KeyBERT và tóm tắt.
3. Ô fit học IDF. Nếu thư mục chỉ có file mẫu trong `samples/` thì giữ file joblib đã có.
4. Ô cuối: máy local đọc `FILE_PATH` hoặc dán `TEXT`. Trên Colab, ô này hiện hộp chọn file `.txt`/`.pdf`. Có `HF_TOKEN` thì in thêm tóm tắt.

KeyBERT dùng model có sẵn, không fine-tune. Local cài trước: `pip install -r requirements-train.txt`.

In [ ]:
# Thư mục các file .txt. Để "data/corpus" nếu mở notebook trong project.
DATA_DIR = "data/corpus"
# Trống = data/stopwords/vietnamese-stopwords.txt cạnh thư mục corpus.
STOPWORDS_PATH = ""
# Trống = models/tfidf_vectorizer.joblib của project.
MODEL_PATH = ""
HF_TOKEN = ""  # điền nếu chưa có token trong môi trường

In [ ]:
from pathlib import Path
import os


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def locate(path_text: str) -> Path:
    raw = Path(path_text).expanduser()
    if raw.exists():
        return raw.resolve()
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidate = base / path_text
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Không thấy {path_text}. Hãy điền đường dẫn đầy đủ.")


if in_colab():
    from google.colab import drive

    drive.mount("/content/drive")
    if str(DATA_DIR).strip() in ("", "data/corpus"):
        DATA_DIR = "/content/drive/MyDrive/CTH625_Nhom3/data/corpus"
    get_ipython().run_line_magic(
        "pip",
        "install -q underthesea scikit-learn sentence-transformers keybert pypdf huggingface_hub joblib",
    )

CORPUS = locate(str(DATA_DIR).strip() or "data/corpus")
if str(STOPWORDS_PATH).strip():
    STOPWORDS_FILE = locate(str(STOPWORDS_PATH).strip())
else:
    STOPWORDS_FILE = CORPUS.parent / "stopwords" / "vietnamese-stopwords.txt"
if not STOPWORDS_FILE.is_file():
    raise FileNotFoundError(f"Không thấy từ dừng: {STOPWORDS_FILE}. Điền STOPWORDS_PATH.")
if str(MODEL_PATH).strip():
    MODEL_FILE = Path(MODEL_PATH).expanduser()
else:
    MODEL_FILE = CORPUS.parent.parent / "models" / "tfidf_vectorizer.joblib"
if str(HF_TOKEN).strip():
    os.environ["HF_TOKEN"] = str(HF_TOKEN).strip()
print("CORPUS =", CORPUS)
print("STOPWORDS =", STOPWORDS_FILE)
print("MODEL =", MODEL_FILE)

## Đọc file và tiền xử lý

In [ ]:
"""Đọc .txt/.pdf, tách từ và lọc từ dừng, từ loại."""

from __future__ import annotations

import re
import unicodedata
from functools import lru_cache
from io import BytesIO
from pathlib import Path

from pypdf import PdfReader
from underthesea import pos_tag, sent_tokenize, word_tokenize

POS_KEEP_PREFIXES = ("N", "V", "A")

SUPPORTED_SUFFIXES = {".txt", ".pdf"}


class InputError(ValueError):
    """Lỗi đọc file đầu vào."""


def read_txt_bytes(data: bytes) -> str:
    for encoding in ("utf-8-sig", "utf-8", "utf-16"):
        try:
            return data.decode(encoding)
        except UnicodeDecodeError:
            continue
    raise InputError("Không đọc được file .txt. Hãy lưu file ở encoding UTF-8.")


def repair_pdf_extracted_text(text: str) -> str:
    """Nối mảnh chữ PDF (mỗi glyph một dòng) thành từ tiếng Việt liền.

    Dòng trống (cột / đoạn) được giữ làm ranh giới, không dính hai cụm khác nhau.
    """
    pieces: list[str] = []
    buf = ""
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            if buf:
                pieces.append(buf)
                buf = ""
            continue
        if not buf:
            buf = line
            continue
        last = buf.split()[-1]
        if len(line) <= 3 or len(last) <= 2:
            buf += line
        else:
            buf += " " + line
    if buf:
        pieces.append(buf)
    return "\n\n".join(pieces).strip()


_VN_LETTER = (
    r"A-Za-zÀÁẢÃẠĂẰẮẲẴẶÂẦẤẨẪẬÈÉẺẼẸÊỀẾỂỄỆÌÍỈĨỊÒÓỎÕỌÔỒỐỔỖỘƠỜỚỞỠỢ"
    r"ÙÚỦŨỤƯỪỨỬỮỰỲÝỶỸỴĐàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợ"
    r"ùúủũụưừứửữựỳýỷỹỵđ"
)
_JOIN_LETTER_NEXT = re.compile(rf"(?<=\s)([{_VN_LETTER}])\s+(?=[{_VN_LETTER}])")
_JOIN_LETTER_PREV = re.compile(rf"(?<=[{_VN_LETTER}])\s+([{_VN_LETTER}])(?=\s|$|[.,;:!?])")
_JOIN_ONSET = re.compile(
    rf"(?<=\s)((?:gi|th|nh|ng|ph|kh|tr|qu|ch|gh|ngh))\s+(?=[{_VN_LETTER}])",
    re.IGNORECASE,
)


def _join_isolated_letters(text: str) -> str:
    """Nối chữ cái đứng một mình do PDF layout ('h ọc' → 'học', 'gia n' → 'gian')."""
    previous = None
    while previous != text:
        previous = text
        text = _JOIN_LETTER_NEXT.sub(r"\1", text)
        text = _JOIN_LETTER_PREV.sub(r"\1", text)
        text = _JOIN_ONSET.sub(r"\1", text)
    return text


def _extract_pdf_page(page) -> str:
    try:
        layout = page.extract_text(extraction_mode="layout") or ""
    except Exception:
        layout = ""
    if layout.strip():
        collapsed = re.sub(r"[ \t]{2,}", " ", layout)
        return _join_isolated_letters(collapsed)
    raw = page.extract_text() or ""
    return repair_pdf_extracted_text(raw)


def read_pdf_bytes(data: bytes) -> str:
    reader = PdfReader(BytesIO(data))
    pages = [_extract_pdf_page(page) for page in reader.pages]
    text = "\n".join(pages).strip()
    if not text:
        raise InputError("File PDF không có lớp văn bản (có thể là bản scan).")
    return text


def read_upload(name: str, data: bytes) -> str:
    suffix = Path(name).suffix.lower()
    if suffix == ".txt":
        return read_txt_bytes(data)
    if suffix == ".pdf":
        return read_pdf_bytes(data)
    raise InputError("Chỉ hỗ trợ .txt và .pdf.")


def read_path(path: str | Path) -> str:
    file_path = Path(path)
    if not file_path.exists():
        raise InputError(f"Không tìm thấy file: {file_path}")
    return read_upload(file_path.name, file_path.read_bytes())


def list_text_files(root: str | Path) -> list[Path]:
    return sorted(path for path in Path(root).rglob("*.txt") if path.is_file())

_PUNCT_OR_DIGIT = re.compile(r"^[\W\d_]+$", re.UNICODE)


def normalize_text(text: str) -> str:
    return unicodedata.normalize("NFC", text).strip()


def tokenize_words(text: str) -> list[str]:
    return word_tokenize(normalize_text(text))


def tokenize_sentences(text: str) -> list[str]:
    sentences = sent_tokenize(normalize_text(text))
    return [s.strip() for s in sentences if s.strip()]


def tag_pos(text: str) -> list[tuple[str, str]]:
    sentences = tokenize_sentences(text)
    if not sentences:
        return pos_tag(normalize_text(text))
    pairs: list[tuple[str, str]] = []
    for sentence in sentences:
        pairs.extend(pos_tag(sentence))
    return pairs


def to_phobert_token(token: str) -> str:
    """PhoBERT / vietnamese-bi-encoder nhận từ đa tiếng nối bằng gạch dưới."""
    return token.strip().replace(" ", "_")


def to_phobert_text(tokens: list[str]) -> str:
    return " ".join(to_phobert_token(t) for t in tokens if t.strip())


MAX_KEYWORD_SYLLABLES = 4
_TECH_SHORT = {"ai", "iot", "nlp", "llm", "rag", "ocr", "pos"}
_ASCII_NOISE = {"cv", "tt", "tp", "vt", "email", "tsp"}
_FUNCTION_IN_COMPOUND = {"và", "của", "cho", "là", "các", "những", "được", "này", "đó"}


def syllable_count(token: str) -> int:
    return len(token.replace("_", " ").split())


def is_technical_ascii(token: str) -> bool:
    """Thuật ngữ ASCII: viết tắt đủ dài (BERT, KHCNTT) hoặc tiếng Anh — không phải CV, TP, Email."""
    if "@" in token or re.search(r"[A-Za-z0-9]\.[\s_]*[A-Za-z0-9]", token):
        return False
    core = re.sub(r"[^\w\-]", "", token.replace("_", ""))
    if len(core) < 2:
        return False
    if not all(ord(char) < 128 for char in core):
        return False
    folded = core.lower()
    if folded in _TECH_SHORT:
        return True
    if folded in _ASCII_NOISE:
        return False
    letters = re.sub(r"[^A-Za-z]", "", core)
    if letters.isupper() and len(letters) < 4:
        return False
    if any(char.isupper() or char.isdigit() for char in core) or "-" in core:
        return True
    return core.isascii() and core.isalpha() and len(core) >= 6


def keep_keyword_unit(token: str) -> bool:
    """Một đơn vị từ khóa: từ tiếng Việt 2–4 tiếng, hoặc thuật ngữ ASCII kỹ thuật."""
    if not token or token.replace("_", "").isdigit():
        return False
    if "@" in token or re.search(r"[A-Za-z0-9]\.[\s_]*[A-Za-z0-9]", token):
        return False
    bits = token.replace("_", " ").split()
    if not bits:
        return False
    if len(bits) >= 2 and bits[0].lower() == bits[1].lower():
        return False
    if is_technical_ascii(token):
        return True
    if any(bit.lower() in _FUNCTION_IN_COMPOUND for bit in bits):
        return False
    count = syllable_count(token)
    return 2 <= count <= MAX_KEYWORD_SYLLABLES


def _stopword_variants(word: str) -> set[str]:
    folded = normalize_text(word).lower()
    variants = {folded, folded.replace(" ", "_"), folded.replace("_", " ")}
    return {item for item in variants if item}


@lru_cache(maxsize=1)
def load_stopwords() -> set[str]:
    if not STOPWORDS_FILE.exists():
        raise FileNotFoundError(f"Thiếu danh sách từ dừng: {STOPWORDS_FILE}")
    words: set[str] = set()
    for line in STOPWORDS_FILE.read_text(encoding="utf-8").splitlines():
        raw = line.strip()
        if not raw:
            continue
        words.update(_stopword_variants(raw))
    return words


def is_content_pos(pos: str) -> bool:
    return bool(pos) and pos[0] in POS_KEEP_PREFIXES


def is_noise_token(token: str) -> bool:
    stripped = token.strip()
    if not stripped:
        return True
    return bool(_PUNCT_OR_DIGIT.match(stripped.replace("_", "")))


def keep_token(token: str, stopwords: set[str] | None = None) -> bool:
    words = stopwords if stopwords is not None else load_stopwords()
    folded = normalize_text(token).lower()
    if is_noise_token(folded):
        return False
    return folded not in words and to_phobert_token(folded) not in words


def content_tokens(text: str, use_pos: bool = True) -> list[tuple[str, str]]:
    """Trả về (token, nhãn POS). Bỏ từ dừng và token rác. `use_pos=True` thì chỉ giữ N/V/A."""
    stopwords = load_stopwords()
    kept: list[tuple[str, str]] = []
    for token, pos in tag_pos(text):
        if not keep_token(token, stopwords):
            continue
        if use_pos and not is_content_pos(pos):
            continue
        kept.append((token, pos))
    return kept


def tokens_as_document(pairs: list[tuple[str, str]]) -> str:
    return " ".join(to_phobert_token(token) for token, _ in pairs)



## TF-IDF

In [ ]:
"""Hai cách lấy từ khóa: TF-IDF và KeyBERT."""

from __future__ import annotations

from pathlib import Path

import joblib
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


NGRAM_RANGE = (1, 2)


def _vectorizer() -> TfidfVectorizer:
    return TfidfVectorizer(
        tokenizer=str.split,
        lowercase=True,
        token_pattern=None,
        ngram_range=NGRAM_RANGE,
        norm="l2",
        use_idf=True,
        smooth_idf=True,
    )


def prepare_document(text: str) -> str:
    return tokens_as_document(content_tokens(text, use_pos=True))


def fit_tfidf(corpus_texts: list[str], save_path: Path | None = None) -> TfidfVectorizer:
    documents = []
    for index, text in enumerate(corpus_texts, start=1):
        prepared = prepare_document(text)
        if prepared:
            documents.append(prepared)
        if len(corpus_texts) >= 20 and (index % 25 == 0 or index == len(corpus_texts)):
            print(f"TF-IDF preprocess {index}/{len(corpus_texts)}")
    if not documents:
        raise ValueError("Corpus sau tiền xử lý bị rỗng, không fit được TF-IDF.")
    model = _vectorizer()
    model.fit(documents)
    path = save_path or MODEL_FILE
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, path)
    return model


def load_tfidf(path: Path | None = None) -> TfidfVectorizer | None:
    file_path = path or MODEL_FILE
    if not file_path.exists():
        return None
    return joblib.load(file_path)


def _adjacent_bigrams(text: str) -> set[str]:
    tokens = [to_phobert_token(token).lower() for token in tokenize_words(text) if token.strip()]
    return {f"{tokens[i]} {tokens[i + 1]}" for i in range(len(tokens) - 1)}


def _keep_tfidf_term(feature: str, adjacent: set[str]) -> bool:
    parts = feature.split()
    if len(parts) == 1:
        return keep_keyword_unit(parts[0])
    if len(parts) != 2:
        return False
    if feature.lower() not in adjacent:
        return False
    return keep_keyword_unit(parts[0]) and keep_keyword_unit(parts[1])


def _top_from_vector(model: TfidfVectorizer, text: str, top_n: int) -> list[tuple[str, float]]:
    """Lấy top-k sau khi bỏ unigram quá ngắn và bigram không kề nhau trên văn bản gốc."""
    prepared = prepare_document(text)
    if not prepared:
        return []
    matrix = model.transform([prepared])
    row = matrix.tocoo()
    scored = list(zip(row.col, row.data))
    scored.sort(key=lambda item: item[1], reverse=True)
    names = model.get_feature_names_out()
    adjacent = _adjacent_bigrams(text)
    results: list[tuple[str, float]] = []
    for index, score in scored:
        feature = names[index]
        if not _keep_tfidf_term(feature, adjacent):
            continue
        results.append((feature.replace("_", " "), float(score)))
        if len(results) >= top_n:
            break
    return results


def extract_tfidf(text: str, top_n: int = 10, vectorizer: TfidfVectorizer | None = None) -> list[tuple[str, float]]:
    """Dùng vectorizer đã fit nếu có; nếu chưa train thì lấy câu làm 'văn bản' để tính IDF nội bộ."""
    model = vectorizer or load_tfidf()
    if model is not None:
        return _top_from_vector(model, text, top_n)

    sentences = tokenize_sentences(text)
    documents = [prepare_document(sentence) for sentence in sentences]
    documents = [doc for doc in documents if doc]
    if not documents:
        documents = [prepare_document(text)]
    if not documents or not documents[0]:
        return []
    fallback = _vectorizer()
    fallback.fit(documents)
    return _top_from_vector(fallback, text, top_n)


## KeyBERT

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

EMBEDDING_MODEL = "bkai-foundation-models/vietnamese-bi-encoder"

_keybert_model = None
MAX_CANDIDATES = 80


def get_keybert():
    global _keybert_model
    if _keybert_model is None:
        from keybert import KeyBERT
        from sentence_transformers import SentenceTransformer

        encoder = SentenceTransformer(EMBEDDING_MODEL)
        _keybert_model = KeyBERT(model=encoder)
    return _keybert_model


def _keep_unigram(token: str) -> bool:
    return keep_keyword_unit(token)


def _analyzer(doc: str) -> list[str]:
    return [token for token in doc.split() if _keep_unigram(token)]


def _score(value: object) -> float:
    item = value.item() if hasattr(value, "item") else value
    return float(item)


def extract_keybert(text: str, top_n: int = 10) -> list[tuple[str, float]]:
    stopwords = load_stopwords()
    tokens = [
        to_phobert_token(token).lower()
        for token in tokenize_words(text)
        if keep_token(token, stopwords)
    ]
    segmented = " ".join(tokens)
    if not segmented:
        return []
    vectorizer = CountVectorizer(
        analyzer=_analyzer,
        lowercase=True,
        max_features=MAX_CANDIDATES,
    )
    pairs = get_keybert().extract_keywords(
        segmented,
        vectorizer=vectorizer,
        top_n=max(top_n * 3, 24),
        use_mmr=True,
        diversity=0.5,
    )
    results: list[tuple[str, float]] = []
    for term, score in pairs:
        unit = to_phobert_token(term)
        if not keep_keyword_unit(unit):
            continue
        results.append((unit.replace("_", " "), _score(score)))
        if len(results) >= top_n:
            break
    return results


## Tóm tắt

In [ ]:
def hf_token() -> str:
    return os.environ.get("HF_TOKEN", "").strip() or os.environ.get("HUGGINGFACEHUB_API_TOKEN", "").strip()


def hf_provider() -> str:
    return os.environ.get("HF_PROVIDER", "").strip() or "featherless-ai"


def llm_model() -> str:
    return os.environ.get("LLM_MODEL", "").strip() or "Qwen/Qwen2.5-7B-Instruct"


SYSTEM_PROMPT = (
    "Bạn là trợ lý tóm tắt văn bản tiếng Việt. "
    "Chỉ dùng thông tin có trong văn bản. Không bịa thêm sự kiện, số liệu, "
    "tên người, tên tổ chức hay địa danh nếu chúng không xuất hiện trong văn bản."
)


def build_user_prompt(text: str, keywords: list[str], n_sentences: int) -> str:
    joined = ", ".join(keywords) if keywords else "(không có từ khóa)"
    return (
        f"Từ khóa cốt lõi (neo ngữ nghĩa): {joined}\n\n"
        f"Văn bản:\n{text}\n\n"
        "Yêu cầu:\n"
        f"- Viết đúng khoảng {n_sentences} câu tiếng Việt, mạch lạc, thành một đoạn.\n"
        "- Mỗi câu phải gắn với ít nhất một từ khóa trong danh sách neo "
        "(dùng đúng ý của từ khóa, không bắt buộc lặp nguyên chữ).\n"
        "- Không thêm số liệu, tên riêng hay sự kiện không có trong văn bản.\n"
        "- Không liệt kê lại từ khóa. Không dùng gạch đầu dòng."
    )


def _chat_huggingface(messages: list[dict], max_tokens: int) -> str:
    from huggingface_hub import InferenceClient

    token = hf_token()
    if not token:
        raise RuntimeError(
            "Chưa có HF_TOKEN. Thêm token vào .env hoặc Streamlit Secrets để gọi LLM."
        )
    client_kwargs = {"token": token}
    provider = hf_provider()
    if provider:
        client_kwargs["provider"] = provider
    client = InferenceClient(**client_kwargs)
    try:
        response = client.chat_completion(
            messages=messages,
            model=llm_model(),
            max_tokens=max_tokens,
            temperature=0.3,
        )
    except Exception as exc:
        text = str(exc)
        if "model_not_supported" in text or "not supported by any provider" in text:
            raise RuntimeError(
                "Hugging Face không gọi được Qwen2.5-7B-Instruct vì chưa bật provider. "
                "Vào https://huggingface.co/settings/inference-providers, bật Featherless AI "
                "(Routed by HF, không cần tài khoản Featherless). "
                "Trong .env đặt HF_PROVIDER=featherless-ai rồi chạy lại Streamlit."
            ) from exc
        raise
    content = response.choices[0].message.content
    if not content:
        raise RuntimeError("Hugging Face API trả về nội dung rỗng.")
    return content.strip()


def summarize(text: str, keywords: list[str], n_sentences: int = 5) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(text, keywords, n_sentences)},
    ]
    max_tokens = min(80 * max(n_sentences, 1), 800)
    return _chat_huggingface(messages, max_tokens)


## Fit IDF

In [ ]:
files = list_text_files(CORPUS)
if not files:
    raise FileNotFoundError(f"Không có file .txt trong {CORPUS}")
samples_dir = (CORPUS / "samples").resolve()
only_samples = CORPUS.name == "samples" or (
    samples_dir.is_dir() and all(path.resolve().is_relative_to(samples_dir) for path in files)
)
if only_samples:
    model = load_tfidf(MODEL_FILE)
    if model is None:
        raise FileNotFoundError("Chưa có file joblib. Thêm văn bản .txt rồi chạy lại ô này.")
    print(f"Corpus chỉ có {len(files)} file mẫu. Giữ joblib đã fit, không ghi đè.")
else:
    model = fit_tfidf([read_path(path) for path in files], save_path=MODEL_FILE)
    print("Đã lưu:", MODEL_FILE)
print("Số văn bản:", len(files))
print("Số term:", len(model.get_feature_names_out()))

## Thử như web

In [ ]:
# Sửa rồi chạy ô này.
METHOD = "tfidf"   # "tfidf" hoặc "keybert"
DO_SUMMARY = True  # cần HF_TOKEN. Không có token thì vẫn in từ khóa.
FILE_PATH = "data/corpus/samples/01_bai_bao.txt"
TEXT = """
"""
try:
    import google.colab
    _colab = True
except ImportError:
    _colab = False
UPLOAD = _colab  # Colab: chọn file. Đổi thành False để dùng FILE_PATH.


def resolve_file(path_text: str) -> str:
    raw = Path(path_text).expanduser()
    if raw.exists():
        return str(raw.resolve())
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidate = base / path_text
        if candidate.exists():
            return str(candidate.resolve())
    return str(raw)


if UPLOAD:
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Chưa chọn file.")
    name, data = next(iter(uploaded.items()))
    text = read_upload(name, data).strip()
    source = name
elif str(FILE_PATH).strip():
    if TEXT.strip():
        print("Có cả file và chữ dán, đang dùng file.")
    source_path = resolve_file(str(FILE_PATH).strip())
    text = read_path(source_path).strip()
    source = Path(source_path).name
elif TEXT.strip():
    text = TEXT.strip()
    source = "văn bản dán"
else:
    raise ValueError("Điền FILE_PATH hoặc dán TEXT.")

if not text:
    raise ValueError("File không có nội dung chữ.")
print("Nguồn:", source)
print("Độ dài:", len(text), "ký tự")
if METHOD == "tfidf":
    keywords = extract_tfidf(text, top_n=10)
elif METHOD == "keybert":
    keywords = extract_keybert(text, top_n=10)
else:
    raise ValueError('METHOD phải là "tfidf" hoặc "keybert".')
print()
print("Từ khóa (" + METHOD + "):")
for term, score in keywords:
    print(f"  {score:.4f}  {term}")
if DO_SUMMARY:
    try:
        summary = summarize(text, [term for term, _ in keywords], n_sentences=5)
        print()
        print("Tóm tắt:")
        print(summary)
    except Exception as exc:
        print()
        print("Tóm tắt:", exc)